# 06｜历史相似行情与 2026 敏感度机制分析

本 Notebook 是诊断分析入口，不修改冻结运行包的信号、阈值、持有参数或输出格式。

它读取：

- 本地米筐 CSI500 现货；
- 03 已生成的含持有期精简八列表；
- 事件型八列表和冻结参考，用于逐列一致性审计；
- 冻结 1545 内部面板，用于机制诊断。

输出包括历史相似行情、市场环境与内部状态分解、多视角同步与确认诊断、固定开发期锚定代理、相似阶段未来表现和本地/远端一致性审计。历史窗口的排序只使用窗口结束时已经可见的信息，后续收益只作为事后评价。

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display

PACKAGE_ROOT = Path('/Users/hzy/Desktop/0817合并查看/最终冻结运行上传包_含零段反转_20260820_1350').resolve()
SRC_ROOT = (PACKAGE_ROOT / 'src').resolve()
if not PACKAGE_ROOT.is_absolute() or not SRC_ROOT.is_absolute():
    raise RuntimeError('06 的包路径必须是绝对路径')
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

def absolute_env(name, default):
    value = os.environ.get(name, default).strip()
    path = Path(value).expanduser().resolve()
    if not path.is_absolute():
        raise RuntimeError(f'{name} 必须是绝对路径：{value}')
    return path

SPOT_PATH = absolute_env('COMPANY_SPOT_PATH', '/Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet')
HOLDING_PATH = absolute_env('HOLDING_EIGHT_PATH', '/Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/holding_period/含持有期八列表.csv')
EVENT_PATH = absolute_env('EVENT_EIGHT_PATH', '/Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/holding_period/最终执行日简表.csv')
EXPECTED_PATH = absolute_env('EXPECTED_EIGHT_PATH', str(PACKAGE_ROOT / 'expected/local_freeze/最终执行日简表_零段反转冻结参考.csv'))
STAGE04_DIR = absolute_env('ANALYSIS_04_OUTPUT_DIR', '/Users/hzy/Desktop/0817合并查看/99_中间归档/本地复现历史/本地复现_最新米筐_远端同格式图与结果对比_20260823_0203')
OUTPUT_DIR = absolute_env('ANALYSIS_06_OUTPUT_DIR', str(PACKAGE_ROOT / '06_历史相似行情与机制分解分析_20260824'))
PANEL_TEXT = os.environ.get('PANEL_PATH', '').strip()
PANEL_PATH = Path(PANEL_TEXT).expanduser().resolve() if PANEL_TEXT else None
if PANEL_PATH is not None and not PANEL_PATH.is_absolute():
    raise RuntimeError('PANEL_PATH 必须是绝对路径')

from reproduce_stage_06 import run_stage_06

print('包目录：', PACKAGE_ROOT)
print('本地米筐现货：', SPOT_PATH)
print('事件型八列表：', EVENT_PATH)
print('含持有期八列表：', HOLDING_PATH)
print('06 输出目录：', OUTPUT_DIR)

包目录： /Users/hzy/Desktop/0817合并查看/最终冻结运行上传包_含零段反转_20260820_1350
本地米筐现货： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet
事件型八列表： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/holding_period/最终执行日简表.csv
含持有期八列表： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/holding_period/含持有期八列表.csv
06 输出目录： /Users/hzy/Desktop/0817合并查看/最终冻结运行上传包_含零段反转_20260820_1350/06_历史相似行情与机制分解分析_20260824


## 1. 运行 06 分析

事件型八列和含持有期八列在审计中分开处理：前者与冻结参考逐列比较，后者只用于连续持有路径和收益分析。

In [2]:
metadata = run_stage_06(
    SPOT_PATH,
    HOLDING_PATH,
    OUTPUT_DIR,
    EXPECTED_PATH,
    STAGE04_DIR,
    PANEL_PATH,
    EVENT_PATH,
)
print(json.dumps(metadata['audit'], ensure_ascii=False, indent=2))
print('历史相似窗口：', metadata['analogue_dates'])
print('图像数量：', len(metadata['figures']))
print('表格数量：', len(metadata['tables']))

[06] 读取本地现货： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet
[06] 读取含持有期八列表： /Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/holding_period/含持有期八列表.csv
[06] 读取冻结参数参考： /Users/hzy/Desktop/0817合并查看/最终冻结运行上传包_含零段反转_20260820_1350/expected/local_freeze/最终执行日简表_零段反转冻结参考.csv
[06] 重建执行日期帧： 2096 2018-01-03 2026-08-24
{
  "spot_path": "/Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/CSI500_SPOT_md_eod_raw_RQ_20260817.parquet",
  "spot_sha256": "e6d25f8777eea7894b93bcaea1f16928efdeeff751213d58ade970beeae09da1",
  "spot_rows": 5255,
  "spot_date_min": "2005-01-04",
  "spot_date_max": "2026-08-21",
  "event_signal_path": "/Users/hzy/Desktop/0817合并查看/99_中间归档/本地数据快照_不入库/_rq_latest_20260824/holding_period/最终执行日简表.csv",
  "local_event_signal_rows": 2096,
  "holding_path_rows": 2096,
  "reference_common_rows": 2094,
  "reference_date_max": "2026-08-20",
  "signal_mismatch_total": 0,
  "signal_mismatch_by_c

## 2. 关键表格

下面的指标只用于诊断和解释，不会回写冻结参数。

In [3]:
tables = OUTPUT_DIR / 'tables'
audit = pd.read_csv(tables / '本地远端一致性审计摘要.csv', encoding='utf-8-sig')
analogs = pd.read_csv(tables / '历史相似行情候选_60日.csv', encoding='utf-8-sig')
annual = pd.read_csv(tables / '年度行情机制与收益对比.csv', encoding='utf-8-sig')
display(audit)
display(analogs)
display(annual.tail(8))

,check,value,pass
0,local event signal values vs freeze reference ...,0 mismatches / 2094 rows,True
1,03 holding path is intentionally expanded from...,not compared cell-for-cell,True
2,04 state/date output matches regenerated execu...,True,True
3,local core metrics vs remote reference,18/18 metrics within tolerance,True
4,latest row has real signal values,2026-08-21 -> 2026-08-24,True
5,signal generation uses no future labels,metadata/engine boundary,True


,label,window_end,rank,return_20,return_60,return_120,vol_20,vol_60,drawdown_60,range_20,...,positive_sync_mean_60,negative_sync_mean_60,neutral_share_60,switch_rate_60,distance,forward_return_20,forward_return_60,forward_vol_20,future_switches_20,future_directional_days_20
0,2026当前窗口,2026-08-21,0,0.042698,-0.082783,-0.092871,0.336310,0.352743,-0.130328,0.023553,...,1.016667,1.983333,0.550000,0.133333,0.000000,NaN,NaN,NaN,NaN,NaN
1,历史相似1,2024-03-13,1,0.126763,-0.026908,-0.051415,0.285308,0.306034,-0.027140,0.023751,...,1.450000,1.833333,0.566667,0.133333,0.796618,-0.025465,-0.040724,0.192049,1.0,0.0
2,历史相似2,2022-10-31,2,0.000136,-0.077375,0.021173,0.233943,0.201888,-0.105705,0.020679,...,1.166667,1.666667,0.633333,0.100000,0.814306,0.042737,0.083009,0.148959,4.0,8.0
3,历史相似3,2020-04-09,3,-0.047780,-0.032252,0.037288,0.327792,0.360875,-0.097539,0.025264,...,1.600000,1.433333,0.683333,0.083333,0.840062,0.035698,0.257641,0.190512,2.0,4.0
4,历史相似4,2022-06-02,4,0.068886,-0.113873,-0.164228,0.204531,0.315873,-0.106616,0.018395,...,1.316667,1.666667,0.683333,0.083333,0.849842,0.059243,0.036183,0.179171,3.0,16.0
5,历史相似5,2018-11-12,5,0.067284,-0.109100,-0.249773,0.309442,0.274779,-0.113463,0.024720,...,1.400000,1.366667,0.833333,0.066667,0.943725,-0.027306,0.006855,0.265709,3.0,5.0
6,历史相似6,2024-07-31,6,-0.004991,-0.115915,-0.008421,0.241999,0.185959,-0.120638,0.016373,...,0.666667,2.283333,0.650000,0.116667,0.974539,-0.079077,0.193477,0.133508,1.0,0.0


,year,phase,market_return_60_pct,market_vol_60_pct,axis_mean_60,axis_std_60,slow_fast_mean_60,positive_sync_mean_60,negative_sync_mean_60,neutral_share_60_pct,switch_rate_60_pct,raw_return_pct,adjusted_return_pct,adjusted_directional_days
1,2019,Development,4.284029,23.474238,0.535628,0.239031,-0.028307,1.885861,0.788934,73.804645,4.733607,44.066750,71.376109,125
2,2020,Development,6.415860,24.786599,0.503965,0.221690,0.019161,1.787517,0.975926,68.395062,6.268861,9.357476,24.654333,112
3,2021,Development,3.197195,15.964020,0.448376,0.229205,0.071584,1.584156,1.262209,71.673525,7.496571,6.589651,9.827413,153
4,2022,Development,-3.893785,21.090914,0.400123,0.233493,-0.047663,1.340220,1.452479,64.662534,9.228650,21.729503,28.805314,140
5,2023,Validation,-2.047459,13.159095,0.449119,0.240287,-0.068176,1.541598,1.203926,64.614325,8.333333,6.642019,4.482964,122
6,2024,Validation,2.626623,26.741246,0.464620,0.254853,0.027779,1.544766,1.326860,69.187328,9.276860,28.581021,59.099990,133
7,2025,Test,5.574919,20.288527,0.570128,0.218060,0.049386,2.067558,0.733265,69.218107,3.401920,9.905724,42.934089,121
8,2026,Test,5.362981,25.793672,0.456175,0.289365,0.028206,1.579978,1.294481,55.670996,8.798701,22.883916,35.478338,82


## 3. 解释边界

图 01—02 用历史相似窗口回答“以前有没有类似行情”；图 03—05 分解市场环境、滚动相对分数、同步性和确认/滞回；图 06—07 比较相似阶段的后续表现及年度覆盖变化。

固定开发期锚定是诊断代理，不是正式冻结逻辑。要完全拆出每个原始因子进入滚动 z-score 前后的反事实，需要额外保存冻结引擎的原始中间特征，但本 Notebook 不因此修改生产引擎。